# **Project Name — Voyage Analytics: End-to-End Travel MLOps**

##### **Project Type** — EDA / Regression / Classification / Recommendation / MLOps

##### **Contribution** — Individual

##### **Team Member 1 — Harsh Paul**

# **Project Summary**

Voyage Analytics examines how a travel provider can turn operational booking data into better pricing, customer insight, and hotel discovery. The project combines three supplied datasets: 271,888 flight records, 40,552 hotel bookings, and 1,340 traveller profiles. They describe routes, ticket classes, flight durations and distances, hotel stays, traveller companies, ages, and historical gender labels. Together they make it possible to study both the economics of a trip and the preferences behind a booking.

The first business use case is flight-price estimation. A price quote should be consistent with route, travel date, cabin type, agency, duration, and distance. The regression workflow engineers calendar and route features, uses a chronological train/test split to avoid training on future observations, and evaluates a Histogram Gradient Boosting model with RMSE, MAE, and R². The resulting sklearn pipeline is saved as a single artifact, so its preprocessing and estimator cannot drift apart when it is called through the Flask API.

The second use case is an educational historic-label classification exercise. A character-level name representation and age are used to reproduce the binary labels present in the supplied dataset. This is explicitly limited to demonstrating the classification workflow and its accuracy/F1 evaluation; names and age are not a reliable or appropriate basis for determining a person’s gender identity or making decisions about people.

The third use case is hotel recommendation. Hotel stays are transformed into an implicit-feedback user–hotel matrix, with repeat stays and longer stays representing stronger affinity. Truncated SVD produces a compact representation of traveller preferences. To avoid evaluating on interactions already seen in training, the last booking for each eligible user is held out and HitRate@5 measures whether the system retrieves it among its top recommendations. The recommendation API excludes previously visited hotels where possible and uses popular hotels as a cold-start fallback.

The project is designed as a deployable MLOps system rather than a notebook-only analysis. MLflow records model parameters and metrics, an Airflow DAG validates data and retrains all models on a schedule, Docker Compose runs the API, dashboard, and tracking server locally, and Kubernetes manifests supply probes, resource limits, a service, and horizontal scaling for the API. A Jenkins pipeline performs installation, training, tests, image builds, and optional deployment. The Streamlit application gives a non-technical user a way to inspect the data, obtain a flight-price quote, and request hotel recommendations. This integrated workflow demonstrates how analytics moves from exploratory insight to reliable, monitored model delivery.

# **GitHub Link**

`https://github.com/harshpaul1999/Voyage-Analytics` (update only if your GitHub username or repository name differs).

# **Problem Statement**

Travel companies need fast, evidence-based price estimates and relevant hotel suggestions, yet booking data is distributed across flights, hotels, and users. The challenge is to transform this historical data into reproducible models and deploy them safely with automated training, testing, tracking, and scalable serving.

#### **Define Your Business Objective?**

Build a reproducible travel intelligence platform that predicts flight prices, demonstrates classification evaluation on the supplied label, and recommends hotels from historical stays. The system must expose models through an API, visualise results in Streamlit, and use Docker, Kubernetes, Airflow, Jenkins, and MLflow to make the lifecycle observable and repeatable.

# **General Guidelines**

- Every transformation used in this notebook is duplicated in `src/` for production inference.
- Charts follow the UBM rule: univariate, bivariate, and multivariate analysis.
- Each visualization states why it was selected, how to read the result, and the business impact.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
if 'google.colab' in sys.modules and not Path('/content/Voyage-Analytics').exists():
    !git clone --depth 1 https://github.com/harshpaul1999/Voyage-Analytics.git /content/Voyage-Analytics
if Path('/content/Voyage-Analytics').exists():
    os.chdir('/content/Voyage-Analytics')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 100)
RANDOM_STATE = 42

### Dataset Loading

In [ ]:
def find_data_dir():
    candidates = [Path('data/raw'), Path('data'), Path('/content/travel_capstone'), Path('../data/raw')]
    for candidate in candidates:
        if all((candidate / f'{name}.csv').exists() for name in ('flights', 'hotels', 'users')):
            return candidate
    if 'google.colab' in sys.modules:
        from google.colab import files
        destination = Path('/content/travel_capstone')
        destination.mkdir(parents=True, exist_ok=True)
        print('Upload flights.csv, hotels.csv, and users.csv.')
        for filename, content in files.upload().items():
            (destination / filename).write_bytes(content)
        return destination
    raise FileNotFoundError('Put the three CSVs in data/raw/, data/, or /content/travel_capstone/.')

DATA_DIR = find_data_dir()
flights = pd.read_csv(DATA_DIR / 'flights.csv', parse_dates=['date'])
hotels = pd.read_csv(DATA_DIR / 'hotels.csv', parse_dates=['date'])
users = pd.read_csv(DATA_DIR / 'users.csv')
print('Loaded data from:', DATA_DIR.resolve())

### Dataset First View

In [ ]:
display(flights.head())
display(hotels.head())
display(users.head())

### Dataset Rows & Columns Count

In [ ]:
overview = pd.DataFrame({'Dataset':['Flights','Hotels','Users'],'Rows':[len(flights),len(hotels),len(users)],'Columns':[flights.shape[1],hotels.shape[1],users.shape[1]]})
display(overview)

### Dataset Information

#### Duplicate Values

#### Missing Values/Null Values

In [ ]:
for name, frame in {'Flights':flights,'Hotels':hotels,'Users':users}.items():
    print(f'\n{name}: duplicates={frame.duplicated().sum()}')
    display(pd.DataFrame({'dtype':frame.dtypes.astype(str),'missing':frame.isna().sum(),'unique':frame.nunique()}))

### What did you know about your dataset?

The supplied data are complete at column level. Flights hold the transaction-level price target and travel attributes, hotels represent historical user–item interactions, and users link bookings to demographic/company information. Dates are parsed before modelling and IDs are not used as flight-price predictors because they identify records rather than describing an available quote.

## ***2. Understanding Your Variables***

### Variables Description

| Dataset | Variables | Meaning |
|---|---|---|
| Users | `code`, `company`, `name`, `gender`, `age` | Traveller profile and historical label |
| Flights | `travelCode`, `userCode`, `from`, `to`, `flightType`, `price`, `time`, `distance`, `agency`, `date` | Flight booking and price target |
| Hotels | `travelCode`, `userCode`, `name`, `place`, `days`, `price`, `total`, `date` | Hotel stay and interaction history |

In [ ]:
for name, frame in {'flights':flights,'hotels':hotels,'users':users}.items():
    print(f'\n{name.upper()} unique values')
    display(frame.nunique().sort_values(ascending=False).to_frame('unique_values'))

## ***3. Data Wrangling***

In [ ]:
flights_eda=flights.drop_duplicates().dropna().copy(); hotels_eda=hotels.drop_duplicates().dropna().copy(); users_eda=users.drop_duplicates().dropna().copy()
for frame in (flights_eda,hotels_eda):
    frame['year']=frame.date.dt.year; frame['month']=frame.date.dt.month
flights_eda['route']=flights_eda['from']+' → '+flights_eda['to']; hotels_eda['hotel_key']=hotels_eda['name']+' | '+hotels_eda['place']
print('Clean shapes:',flights_eda.shape,hotels_eda.shape,users_eda.shape)

### What manipulations have you done and insights you found?

The workflow removes exact duplicates, parses dates, derives calendar features, and creates a direction-sensitive flight route plus a hotel key that distinguishes the same hotel name in different cities. No values were imputed because the supplied columns contain no missing values. These transformations are reproduced in `src/features/` so notebook exploration and deployed inference use consistent definitions.

## ***4. Data Visualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart 1 — Missing-value audit

##### 1. Why did you pick the specific chart?
A bar chart makes data-completeness issues comparable.

##### 2. What is/are the insight(s) found from the chart?
Zero-height bars confirm no imputation is required.

##### 3. Will the gained insights help creating a positive business impact?
Future non-zero bars should trigger data-quality alerts.

In [ ]:
pd.concat({'flights':flights.isna().sum(),'hotels':hotels.isna().sum(),'users':users.isna().sum()},axis=1).fillna(0).plot(kind='bar',figsize=(12,4)); plt.title('Missing values by column'); plt.show()

#### Chart 2 — Traveller age distribution

##### 1. Why did you pick the specific chart?
A histogram shows spread and outliers of a numeric variable.

##### 2. What is/are the insight(s) found from the chart?
The shape identifies the most common traveller age bands.

##### 3. Will the gained insights help creating a positive business impact?
It supports aggregate segmentation, not individual decisions.

In [ ]:
sns.histplot(users_eda,x='age',bins=20,kde=True); plt.title('Traveller age distribution'); plt.show()

#### Chart 3 — Historical label balance

##### 1. Why did you pick the specific chart?
A count plot checks class imbalance before classification.

##### 2. What is/are the insight(s) found from the chart?
Bar heights show whether weighted evaluation is needed.

##### 3. Will the gained insights help creating a positive business impact?
Accuracy must be interpreted with weighted F1.

In [ ]:
sns.countplot(data=users_eda,x='gender'); plt.title('Historical label counts'); plt.show()

#### Chart 4 — Top traveller companies

##### 1. Why did you pick the specific chart?
A ranking exposes corporate-travel concentration.

##### 2. What is/are the insight(s) found from the chart?
Leading companies have the most traveller profiles.

##### 3. Will the gained insights help creating a positive business impact?
High-volume accounts can be prioritised for service design.

In [ ]:
users_eda.company.value_counts().head(12).sort_values().plot(kind='barh'); plt.title('Top companies'); plt.show()

#### Chart 5 — Flight-price distribution

##### 1. Why did you pick the specific chart?
A histogram reveals price spread and skew.

##### 2. What is/are the insight(s) found from the chart?
A right tail identifies expensive bookings.

##### 3. Will the gained insights help creating a positive business impact?
MAE and RMSE both matter for costly quote errors.

In [ ]:
sns.histplot(flights_eda,x='price',bins=60,kde=True); plt.title('Flight-price distribution'); plt.show()

#### Chart 6 — Flight-duration distribution

##### 1. Why did you pick the specific chart?
Duration is a core pricing input.

##### 2. What is/are the insight(s) found from the chart?
Most records concentrate in common domestic-duration bands.

##### 3. Will the gained insights help creating a positive business impact?
Rare long trips should be monitored for error concentration.

In [ ]:
sns.histplot(flights_eda,x='time',bins=50); plt.title('Flight duration'); plt.show()

#### Chart 7 — Flight-distance distribution

##### 1. Why did you pick the specific chart?
Distance is a direct travel-cost proxy.

##### 2. What is/are the insight(s) found from the chart?
The shape highlights common route lengths and tails.

##### 3. Will the gained insights help creating a positive business impact?
Outlier distance values deserve validation.

In [ ]:
sns.histplot(flights_eda,x='distance',bins=50); plt.title('Flight distance'); plt.show()

#### Chart 8 — Hotel booking totals

##### 1. Why did you pick the specific chart?
Totals represent hotel revenue exposure.

##### 2. What is/are the insight(s) found from the chart?
Right skew indicates a small number of high-value stays.

##### 3. Will the gained insights help creating a positive business impact?
Revenue concentration informs inventory planning.

In [ ]:
sns.histplot(hotels_eda,x='total',bins=50); plt.title('Hotel booking totals'); plt.show()

#### Chart 9 — Hotel stay length

##### 1. Why did you pick the specific chart?
A count plot is suitable for discrete day counts.

##### 2. What is/are the insight(s) found from the chart?
Short stays dominate while long stays are rarer.

##### 3. Will the gained insights help creating a positive business impact?
Stay length lightly strengthens recommendation affinity.

In [ ]:
sns.countplot(data=hotels_eda,x='days'); plt.title('Hotel stay length'); plt.show()

#### Chart 10 — Flight bookings by ticket type

##### 1. Why did you pick the specific chart?
This compares the categorical market mix.

##### 2. What is/are the insight(s) found from the chart?
It shows training support for each ticket type.

##### 3. Will the gained insights help creating a positive business impact?
High-volume ticket types need consistently reliable quotes.

In [ ]:
sns.countplot(data=flights_eda,x='flightType'); plt.title('Bookings by ticket type'); plt.show()

#### Chart 11 — Flight bookings by agency

##### 1. Why did you pick the specific chart?
This shows distribution across sales channels.

##### 2. What is/are the insight(s) found from the chart?
Channel volume differs across agencies.

##### 3. Will the gained insights help creating a positive business impact?
Agency is a categorical price feature.

In [ ]:
sns.countplot(data=flights_eda,x='agency'); plt.title('Bookings by agency'); plt.show()

#### Chart 12 — Price by ticket type

##### 1. Why did you pick the specific chart?
A box plot compares price distributions and outliers.

##### 2. What is/are the insight(s) found from the chart?
Higher ticket tiers should have higher medians.

##### 3. Will the gained insights help creating a positive business impact?
Tier-aware predictions support packaging and revenue management.

In [ ]:
sns.boxplot(data=flights_eda,x='flightType',y='price'); plt.ylim(0,flights_eda.price.quantile(.99)); plt.title('Price by ticket type'); plt.show()

#### Chart 13 — Price by agency

##### 1. Why did you pick the specific chart?
A box plot reveals channel-level variation.

##### 2. What is/are the insight(s) found from the chart?
Median and spread may differ by agency.

##### 3. Will the gained insights help creating a positive business impact?
Channel effects can guide partner discussions, not causal claims.

In [ ]:
sns.boxplot(data=flights_eda,x='agency',y='price'); plt.ylim(0,flights_eda.price.quantile(.99)); plt.title('Price by agency'); plt.show()

#### Chart 14 — Price versus distance

##### 1. Why did you pick the specific chart?
A scatterplot tests two continuous commercial variables.

##### 2. What is/are the insight(s) found from the chart?
An upward pattern indicates distance matters but is insufficient alone.

##### 3. Will the gained insights help creating a positive business impact?
Multifeature pricing avoids a simplistic distance-only rule.

In [ ]:
flight_sample=flights_eda.sample(min(10000,len(flights_eda)),random_state=RANDOM_STATE); sns.scatterplot(data=flight_sample,x='distance',y='price',alpha=.25,s=12); plt.title('Price vs distance'); plt.show()

#### Chart 15 — Duration versus distance by ticket type

##### 1. Why did you pick the specific chart?
Colour adds a third variable to the relationship.

##### 2. What is/are the insight(s) found from the chart?
Similar routes can have different durations and ticket types.

##### 3. Will the gained insights help creating a positive business impact?
The model benefits from both numeric and categorical inputs.

In [ ]:
sns.scatterplot(data=flight_sample,x='distance',y='time',hue='flightType',alpha=.35,s=14); plt.title('Duration vs distance'); plt.show()

#### Chart 16 — Highest-priced routes

##### 1. Why did you pick the specific chart?
Route ranking surfaces high-value corridors.

##### 2. What is/are the insight(s) found from the chart?
Top median-price routes deserve closer monitoring.

##### 3. Will the gained insights help creating a positive business impact?
Revenue management can prioritise important corridors.

In [ ]:
flights_eda.groupby('route').price.median().sort_values(ascending=False).head(12).sort_values().plot(kind='barh'); plt.title('Top routes by median price'); plt.show()

#### Chart 17 — Monthly average flight price

##### 1. Why did you pick the specific chart?
A line chart reveals temporal change.

##### 2. What is/are the insight(s) found from the chart?
Monthly movement signals possible calendar effects.

##### 3. Will the gained insights help creating a positive business impact?
Seasonality supports scheduled retraining and capacity planning.

In [ ]:
flights_eda.set_index('date').resample('MS').price.mean().plot(figsize=(11,4)); plt.title('Monthly average flight price'); plt.show()

#### Chart 18 — Hotel bookings by destination

##### 1. Why did you pick the specific chart?
Destination counts define the popularity baseline.

##### 2. What is/are the insight(s) found from the chart?
The leading places receive the most bookings.

##### 3. Will the gained insights help creating a positive business impact?
Popular fallbacks help recommendation cold start.

In [ ]:
hotels_eda.place.value_counts().head(12).sort_values().plot(kind='barh'); plt.title('Hotel bookings by destination'); plt.show()

#### Chart 19 — Hotel daily price versus total

##### 1. Why did you pick the specific chart?
This tests the expected bill relationship.

##### 2. What is/are the insight(s) found from the chart?
Total value grows with nightly price and stay length.

##### 3. Will the gained insights help creating a positive business impact?
It validates the source and separates revenue from affinity.

In [ ]:
hotel_sample=hotels_eda.sample(min(8000,len(hotels_eda)),random_state=RANDOM_STATE); sns.scatterplot(data=hotel_sample,x='price',y='total',hue='days',alpha=.35,legend=False); plt.title('Hotel daily price vs total'); plt.show()

#### Chart 20 — Stay days versus total by destination

##### 1. Why did you pick the specific chart?
A multivariate box plot compares duration and place.

##### 2. What is/are the insight(s) found from the chart?
Longer stays increase totals while cities have different price levels.

##### 3. Will the gained insights help creating a positive business impact?
Recommendations learn preference from stays rather than raw spend.

In [ ]:
top_places=hotels_eda.place.value_counts().head(5).index; sns.boxplot(data=hotels_eda[hotels_eda.place.isin(top_places)],x='days',y='total',hue='place'); plt.ylim(0,hotels_eda.total.quantile(.98)); plt.title('Stay total by duration and place'); plt.show()

#### Chart 21 — Correlation heatmap

##### 1. Why did you pick the specific chart?
A heatmap summarises numeric relationships.

##### 2. What is/are the insight(s) found from the chart?
Distance, duration, and price associations can be checked together.

##### 3. Will the gained insights help creating a positive business impact?
It informs feature selection but does not establish causality.

In [ ]:
sns.heatmap(flights_eda[['price','time','distance','year','month']].corr(),annot=True,cmap='coolwarm',center=0); plt.title('Flight numeric correlations'); plt.show()

#### Chart 22 — Pair plot sample

##### 1. Why did you pick the specific chart?
A pair plot scans multivariate structure.

##### 2. What is/are the insight(s) found from the chart?
Clusters and non-linearity motivate gradient boosting.

##### 3. Will the gained insights help creating a positive business impact?
It helps identify data issues before deployment.

In [ ]:
sns.pairplot(flights_eda.sample(min(2000,len(flights_eda)),random_state=RANDOM_STATE)[['price','time','distance','flightType']],hue='flightType',corner=True,plot_kws={'alpha':.35,'s':12}); plt.show()

## ***5. Hypothesis Testing***

**Hypothesis:** flight ticket type is associated with flight price. The box plot and out-of-sample model results provide practical evidence. A formal extension can apply ANOVA or a non-parametric test after checking assumptions. Because the goal is prediction, held-out RMSE/MAE/R² are the primary acceptance measures.

## ***6. Feature Engineering & Data Pre-processing***

In [ ]:
from src.features.flight_features import prepare_flight_features
X_flight=prepare_flight_features(flights_eda); y_flight=flights_eda.loc[X_flight.index,'price']
print('Flight feature shape:',X_flight.shape); display(X_flight.head())

## ***7. Model Implementation***

### Regression: Flight price
A chronological 80/20 holdout, one-hot categorical encoding, date features, and Histogram Gradient Boosting model non-linear price effects. Record IDs are excluded to prevent memorisation.

### Classification: Historic dataset label
Character n-grams from a first name plus scaled age feed a Linear SVC. This is an educational reproduction of the supplied label only, never a determination of gender identity.

### Recommendation: Hotel suggestions
Historical stays become implicit affinities. Truncated SVD learns user-hotel preference patterns and the most recent booking is held out to evaluate HitRate@5.

In [ ]:
for command in [[sys.executable,'-m','src.models.train_flight_price_model'],[sys.executable,'-m','src.models.train_gender_model'],[sys.executable,'-m','src.models.train_recommendation_model']]:
    print('Running:',' '.join(command)); subprocess.run(command,check=True)

## ***8. Model Performance & Evaluation***

In [ ]:
metrics={path.stem:json.loads(path.read_text()) for path in Path('reports').glob('*_metrics.json')}
display(pd.DataFrame(metrics).T)

### Evaluation interpretation

- Lower **RMSE** and **MAE** and higher **R²** mean more accurate future flight-price estimates.
- Higher **accuracy** and **weighted F1** mean closer reproduction of the historic training label, with the responsible-use limitation above.
- Higher **HitRate@5** means more held-out last bookings are recovered in five recommendations.

## ***9. MLOps & Deployment***

The Flask API exposes `/health`, `/predict_flight_price`, `/predict_gender`, and `/recommend`. Docker Compose starts the API, Streamlit dashboard, and MLflow server. The Airflow DAG validates data, retrains all models, and runs tests daily. Jenkins implements CI/CD and Kubernetes adds rolling deployment, probes, resource limits, services, and API autoscaling.

## ***10. Conclusion***

Voyage Analytics moves from descriptive travel analysis to deployable pricing and recommendation services. Next steps are model monitoring, drift alerts, explicit traveller feedback, qualified fairness review, secured configuration, and a shared remote model registry.